# UN General Debate NLP & Topic Modeling — Africa

This notebook is the clean, reproducible version of the original master's coursework analysis. It focuses on African UN General Debate speeches and separates descriptive NLP from topic-model estimation.

**Interpretation boundary:** this is descriptive text analysis; no causal claims are made.

## 1. Setup

Install dependencies with `pip install -r requirements.txt`, then download NLTK resources once with `python -m nltk.downloader punkt stopwords wordnet`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.preprocessing import add_text_features, filter_africa, load_ungd
from src.sentiment import add_sentiment_features
from src.topic_models import (
    bertopic_word_lists,
    fit_bertopic,
    fit_lda,
    fit_nmf,
    lda_topic_prevalence,
    tokenize_for_coherence,
    topic_coherence,
)
from src.visualization import plot_speeches_over_time, plot_word_count_distribution

DATA_PATH = ROOT / "data" / "un-general-debates.csv"


## 2. Load and construct the Africa sample

The final coursework notebook records 7,507 speeches in the full corpus (1970–2015) and 2,159 speeches in the Africa subset.

In [ ]:
ungd = load_ungd(DATA_PATH)
africa = filter_africa(ungd)
africa = add_text_features(africa)

summary = pd.Series({
    "full_corpus_speeches": len(ungd),
    "africa_speeches": len(africa),
    "africa_country_codes": africa["country"].nunique(),
    "first_year": int(africa["year"].min()),
    "last_year": int(africa["year"].max()),
    "median_words": float(africa["word_count"].median()),
})
summary


## 3. Corpus diagnostics

These diagnostics describe the available sample. The number of speeches per year should not be interpreted as a direct measure of diplomatic participation because membership and corpus coverage change over time.

In [ ]:
fig, ax = plot_speeches_over_time(africa)
fig


In [ ]:
fig, ax = plot_word_count_distribution(africa)
fig


## 4. Sentiment — corrected sentence-level VADER

The original exploratory notebook applied sentence tokenization after reducing speeches to individual tokens. This rebuild instead scores sentence-like units from the original speech text and aggregates them to the speech level.

In [ ]:
africa_sentiment = add_sentiment_features(africa)
sentiment_by_year = (
    africa_sentiment.groupby("year", as_index=False)["sentiment_compound"].mean()
)
sentiment_by_year.head()


## 5. LDA

The specification preserves the main coursework settings: 10 topics, 50 passes, fixed random seed, and vocabulary filtering.

In [ ]:
documents = africa["processed_text"].tolist()
lda = fit_lda(documents, num_topics=10, passes=50, random_state=0)
print(f"LDA c_v coherence: {lda.coherence:.4f}")

lda_topics = {
    topic_id: [word for word, _ in lda.model.show_topic(topic_id, topn=10)]
    for topic_id in range(lda.model.num_topics)
}
lda_prevalence = pd.DataFrame({
    "topic": range(lda.model.num_topics),
    "prevalence": lda_topic_prevalence(lda),
    "top_words": [", ".join(lda_topics[i][:5]) for i in range(lda.model.num_topics)],
}).sort_values("prevalence", ascending=False)
lda_prevalence


## 6. NMF

NMF is estimated on a TF–IDF document-term matrix. In the rebuild, its topic words are evaluated against the same tokenized reference corpus used for coherence comparisons.

In [ ]:
nmf = fit_nmf(documents, num_topics=10, random_state=0)
print(f"NMF c_v coherence: {nmf.coherence:.4f}")
pd.DataFrame({
    "topic": range(len(nmf.topics)),
    "top_words": [", ".join(words) for words in nmf.topics],
})


## 7. BERTopic (optional)

BERTopic is computationally heavier and may download a sentence-transformer model on first run. The cell is off by default so the classical pipeline remains lightweight.

In [ ]:
RUN_BERTOPIC = False

if RUN_BERTOPIC:
    bertopic_model, bertopic_assignments, probabilities = fit_bertopic(
        documents,
        n_gram_range=(1, 3),
        min_topic_size=10,
    )
    bertopic_topics = bertopic_word_lists(bertopic_model, top_n=10)
    reference_tokens = tokenize_for_coherence(documents)
    bertopic_cv = topic_coherence(bertopic_topics, reference_tokens)
    print(f"BERTopic c_v coherence: {bertopic_cv:.4f}")
else:
    print("BERTopic skipped. Set RUN_BERTOPIC = True to estimate it.")


## 8. Interpretation and limitations

- Topic models recover statistical patterns in word co-occurrence; topics require substantive validation with representative speeches.
- Coherence is a diagnostic, not a sufficient criterion for selecting the substantively best model.
- Sentiment in diplomatic language is especially sensitive to preprocessing and should be treated as descriptive.
- The corpus is an observational text archive. This notebook does not estimate causal effects.

See [`docs/methodology.md`](../docs/methodology.md) for the reconciliation of the original coursework files and the methodological changes made in this rebuild.